In [14]:
import cv2
import mediapipe as mp
import numpy as np

# === CONFIG ===
VIDEO_PATH = "time (5).MP4"       # 🔹 Change this to your video file path
OUTPUT_NPY = "output_landmarks.npy" # 🔹 Where you want to save the landmarks
OUTPUT_NPY_1 = "Hand_LandMark/are you hiding something_3.npy" # 🔹 Where you want to save the landmarks


# === MEDIAPIPE SETUP ===
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,     # For video (continuous)
    max_num_hands=1,             # One hand (you can set to 2 if needed)
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# === VIDEO CAPTURE ===
cap = cv2.VideoCapture(VIDEO_PATH)

# List to store all frames' landmarks
all_landmarks = []

while cap.isOpened():
    ret, frame = cap.read()
    if not ret:
        break

    # Convert BGR -> RGB
    frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
    results = hands.process(frame_rgb)

    # Check if hand detected
    if results.multi_hand_landmarks:
        # Take the first hand
        hand_landmarks = results.multi_hand_landmarks[0]
        # Extract landmark coordinates
        frame_landmarks = []
        for lm in hand_landmarks.landmark:
            frame_landmarks.extend([lm.x, lm.y, lm.z])
        all_landmarks.append(frame_landmarks)

    # OPTIONAL: visualize (press Q to quit)
    # mp.solutions.drawing_utils.draw_landmarks(frame, hand_landmarks, mp_hands.HAND_CONNECTIONS)
    # cv2.imshow("Frame", frame)
    # if cv2.waitKey(1) & 0xFF == ord('q'):
    #     break

cap.release()
# cv2.destroyAllWindows()

# Convert to NumPy array
all_landmarks = np.array(all_landmarks)
print(all_landmarks)
print("✅ Saved landmarks shape:", all_landmarks.shape)

# Save as .npy
np.save(OUTPUT_NPY, all_landmarks)
print(f"🎯 Landmarks saved to: {OUTPUT_NPY}")



[[ 4.88736987e-01  8.53759110e-01  2.64901434e-09 ...  5.04610717e-01
   9.69040394e-01 -2.19944678e-02]
 [ 4.77197438e-01  8.51803839e-01 -5.52991430e-08 ...  5.03395855e-01
   9.73434985e-01 -1.84782706e-02]
 [ 4.79132563e-01  8.51077318e-01 -2.82451786e-08 ...  5.07867873e-01
   9.73743141e-01 -2.19236743e-02]
 ...
 [ 5.39927185e-01  8.29513371e-01  1.97039157e-07 ...  5.25455058e-01
   9.45492864e-01 -1.12738293e-02]
 [ 5.41573405e-01  8.38907063e-01  2.00831892e-07 ...  5.29960573e-01
   9.53432918e-01 -1.37302568e-02]
 [ 5.38900137e-01  8.46355200e-01  2.25790444e-07 ...  5.28889537e-01
   9.66393232e-01 -1.48095740e-02]]
✅ Saved landmarks shape: (82, 63)
🎯 Landmarks saved to: output_landmarks.npy


In [20]:
import os
import cv2
import mediapipe as mp
import numpy as np
import csv

# === CONFIG ===
DATASET_DIR = "D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/Videos_Sentence_Level"  # Change to your dataset folder
OUTPUT_NPY_DIR = "Hand_LandMark"
LABELS_CSV = "labels.csv"

# Create output folder if it doesn't exist
os.makedirs(OUTPUT_NPY_DIR, exist_ok=True)

# Mediapipe setup
mp_hands = mp.solutions.hands
hands = mp_hands.Hands(
    static_image_mode=False,
    max_num_hands=1,  # 1 hand per video
    min_detection_confidence=0.5,
    min_tracking_confidence=0.5
)

# Prepare CSV
csv_rows = []
csv_rows.append(["npy_filename", "label"])

# Traverse dataset
for label_name in sorted(os.listdir(DATASET_DIR)):
    label_path = os.path.join(DATASET_DIR, label_name)
    if not os.path.isdir(label_path):
        continue

    print(f"Processing label: {label_name}")

    for video_name in sorted(os.listdir(label_path)):
        video_path = os.path.join(label_path, video_name)
        if not video_path.lower().endswith((".mp4", ".avi", ".mov")):
            continue

        print(f"  Processing video: {video_name}")

        cap = cv2.VideoCapture(video_path)
        all_landmarks = []

        while cap.isOpened():
            ret, frame = cap.read()
            if not ret:
                break

            # Resize if rotated (optional)
            h, w = frame.shape[:2]
            if (w, h) == (320, 240):
                frame = cv2.rotate(frame, cv2.ROTATE_90_CLOCKWISE)

            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            results = hands.process(frame_rgb)

            if results.multi_hand_landmarks:
                hand_landmarks = results.multi_hand_landmarks[0]
                frame_landmark = []
                for lm in hand_landmarks.landmark:
                    frame_landmark.extend([lm.x, lm.y, lm.z])
                all_landmarks.append(frame_landmark)

        cap.release()

        # Handle videos with no landmarks detected
        if len(all_landmarks) == 0:
            print("    ❌ No landmarks found, skipping.")
            continue

        landmarks_array = np.array(all_landmarks)  # (frames, 63)

        # Compute statistical features
        mean_features = np.mean(landmarks_array, axis=0)
        std_features = np.std(landmarks_array, axis=0)
        min_features = np.min(landmarks_array, axis=0)
        max_features = np.max(landmarks_array, axis=0)

        all_features = np.concatenate([mean_features, std_features, min_features, max_features])  # (252,)

        # Save .npy
        npy_filename = f"{label_name}_{os.path.splitext(video_name)[0]}.npy"
        np.save(os.path.join(OUTPUT_NPY_DIR, npy_filename), all_features)

        # Save label row
        csv_rows.append([npy_filename, label_name])

# Save labels.csv
with open(LABELS_CSV, mode="w", newline="") as f:
    writer = csv.writer(f)
    writer.writerows(csv_rows)

print("✅ Done generating npy files and labels.csv.")


Processing label: He is going into the room
  Processing video: MVI_6503.MP4
  Processing video: he is going into the room (2).mp4
  Processing video: he is going into the room.mp4
  Processing video: room (2).MP4
  Processing video: room (3).MP4
  Processing video: room (4).MP4
  Processing video: room.MP4
Processing label: No need to worry dont worry
  Processing video: MVI_6496.MP4
Processing label: This place is beautiful
  Processing video: MVI_6502.MP4
  Processing video: beautiful (2).MP4
  Processing video: beautiful (3).MP4
  Processing video: beautiful (4).MP4
  Processing video: beautiful.MP4
  Processing video: this place is beautiful (2).mp4
  Processing video: this place is beautiful.mp4
Processing label: are you free today
  Processing video: are you free today (2).mp4
  Processing video: are you free today.mp4
  Processing video: free (2).MP4
  Processing video: free (3).MP4
  Processing video: free (4).MP4
  Processing video: free (5).MP4
  Processing video: free.MP4
P

In [22]:
import os
import numpy as np

NpyFolder = "Hand_LandMark"  # Adjust to your folder name

files = sorted(os.listdir(NpyFolder))

count_checked = 0

for f in files:
    if not f.endswith(".npy"):
        continue

    npy_path = os.path.join(NpyFolder, f)
    arr = np.load(npy_path)
    
    print(f"{f}: shape = {arr.shape}")

    count_checked += 1
    if count_checked >= 100:
        break

print("✅ Done checking first 10 files.")



He is going into the room_MVI_6503.npy: shape = (252,)
He is going into the room_he is going into the room (2).npy: shape = (252,)
He is going into the room_he is going into the room.npy: shape = (252,)
He is going into the room_room (2).npy: shape = (252,)
He is going into the room_room (3).npy: shape = (252,)
He is going into the room_room (4).npy: shape = (252,)
He is going into the room_room.npy: shape = (252,)
No need to worry dont worry_MVI_6496.npy: shape = (252,)
This place is beautiful_MVI_6502.npy: shape = (252,)
This place is beautiful_beautiful (2).npy: shape = (252,)
This place is beautiful_beautiful (3).npy: shape = (252,)
This place is beautiful_beautiful (4).npy: shape = (252,)
This place is beautiful_beautiful.npy: shape = (252,)
This place is beautiful_this place is beautiful (2).npy: shape = (252,)
This place is beautiful_this place is beautiful.npy: shape = (252,)
are you free today_are you free today (2).npy: shape = (252,)
are you free today_are you free today.npy

✅ Loaded 687 samples.
✅ Label distribution: Counter({'serve the food': 9, 'can i help you': 8, 'go and sleep': 8, 'i enjoyed a lot': 8, 'i like you i love you': 8, 'speak softly': 8, 'you are bad': 8, 'He is going into the room': 7, 'This place is beautiful': 7, 'are you free today': 7, 'are you hiding something': 7, 'bring water for me': 7, 'can you repeat that please': 7, 'comb your hair': 7, 'congratulations': 7, 'could you please talk slower': 7, 'do me a favour': 7, 'do not hurt me': 7, 'do not make me angry': 7, 'do not take it to the heart': 7, 'do not worry': 7, 'do you need something': 7, 'had your food': 7, 'he came by train': 7, 'he she is my friend': 7, 'he would be coming today': 7, 'help me': 7, 'hi how are you': 7, 'how can i help you': 7, 'how can i trust you': 7, 'how dare you': 7, 'how old are you': 7, 'i am afraid of that': 7, 'i am crying': 7, 'i am feeling bored': 7, 'i am feeling cold': 7, 'i am fine. thank you sir': 7, 'i am hungry': 7, 'i am in dilemma what to d

ValueError: The least populated class in y has only 1 member, which is too few. The minimum number of groups for any class cannot be less than 2.

In [8]:
# from collections import Counter
# 
# # Count labels
# label_counts = Counter(y)
# print("Before filtering:", label_counts)
# 
# # Keep only labels with at least 2 samples
# X_clean = []
# y_clean = []
# 
# for features, label in zip(X, y):
#     if label_counts[label] >= 2:
#         X_clean.append(features)
#         y_clean.append(label)
# 
# # Check again
# print("✅ After filtering:")
# print(Counter(y_clean))
# 
# # Replace X and y
# X = X_clean
# y = y_clean


Before filtering: Counter({'serve the food': 9, 'can i help you': 8, 'go and sleep': 8, 'i enjoyed a lot': 8, 'i like you i love you': 8, 'speak softly': 8, 'you are bad': 8, 'He is going into the room': 7, 'This place is beautiful': 7, 'are you free today': 7, 'are you hiding something': 7, 'bring water for me': 7, 'can you repeat that please': 7, 'comb your hair': 7, 'congratulations': 7, 'could you please talk slower': 7, 'do me a favour': 7, 'do not hurt me': 7, 'do not make me angry': 7, 'do not take it to the heart': 7, 'do not worry': 7, 'do you need something': 7, 'had your food': 7, 'he came by train': 7, 'he she is my friend': 7, 'he would be coming today': 7, 'help me': 7, 'hi how are you': 7, 'how can i help you': 7, 'how can i trust you': 7, 'how dare you': 7, 'how old are you': 7, 'i am afraid of that': 7, 'i am crying': 7, 'i am feeling bored': 7, 'i am feeling cold': 7, 'i am fine. thank you sir': 7, 'i am hungry': 7, 'i am in dilemma what to do': 7, 'i am not really su

In [10]:
import os
import numpy as np
import csv
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from tensorflow.keras import models, layers
from tensorflow.keras.callbacks import EarlyStopping
from sklearn.metrics import classification_report

# === CONFIG ===
NpyFolder = "ML/Hand_LandMark"  # Folder containing your .npy files
LabelsFile = "labels.csv"    # Your labels CSV

# === Load Data ===
X = []
y = []

with open(LabelsFile, mode="r") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        npy_filename, label = row
        npy_path = os.path.join(NpyFolder, npy_filename)

        if not os.path.exists(npy_path):
            print(f"Missing file: {npy_path}")
            continue

        features = np.load(npy_path)  # e.g., shape (1260,)
        X.append(features)
        y.append(label)

print("✅ Loaded", len(X), "samples.")
print("✅ Label distribution before filtering:", Counter(y))

# === Remove labels with fewer than 2 samples ===
label_counts = Counter(y)
X_clean = []
y_clean = []

for features, label in zip(X, y):
    if label_counts[label] >= 2:
        X_clean.append(features)
        y_clean.append(label)

X = X_clean
y = y_clean

print("✅ Label distribution after filtering:", Counter(y))

# === Encode labels ===
le = LabelEncoder()
y_encoded = le.fit_transform(y)
num_classes = len(le.classes_)
print("✅ Encoded labels into", num_classes, "classes.")

# === Scale features ===
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# === Train/test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_encoded,
    test_size=0.2,
    stratify=y_encoded,
    random_state=42
)

print("✅ Train size:", len(X_train), "Test size:", len(X_test))
print("✅ Input feature shape:", X_train.shape)

# === Build Neural Network ===
model = models.Sequential([
    layers.Input(shape=(X_train.shape[1],)),  # e.g., 1260
    layers.Dense(256, activation="relu"),
    layers.Dropout(0.3),
    layers.Dense(128, activation="relu"),
    layers.Dropout(0.2),
    layers.Dense(num_classes, activation="softmax")
])

model.compile(
    optimizer="adam",
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"]
)

early_stop = EarlyStopping(
    monitor="val_loss",
    patience=10,
    restore_best_weights=True
)

# === Train ===
history = model.fit(
    X_train, y_train,
    validation_split=0.2,
    epochs=100,
    batch_size=16,
    callbacks=[early_stop],
    verbose=1
)

# === Evaluate ===
loss, acc = model.evaluate(X_test, y_test)
print(f"🎯 Test Accuracy: {acc:.4f}")

# === Predict and report ===
pred_probs = model.predict(X_test)
pred_classes = pred_probs.argmax(axis=1)

print("🧠 Classification Report:")
print(classification_report(y_test, pred_classes, target_names=le.classes_))


✅ Loaded 687 samples.
✅ Label distribution before filtering: Counter({'serve the food': 9, 'can i help you': 8, 'go and sleep': 8, 'i enjoyed a lot': 8, 'i like you i love you': 8, 'speak softly': 8, 'you are bad': 8, 'He is going into the room': 7, 'This place is beautiful': 7, 'are you free today': 7, 'are you hiding something': 7, 'bring water for me': 7, 'can you repeat that please': 7, 'comb your hair': 7, 'congratulations': 7, 'could you please talk slower': 7, 'do me a favour': 7, 'do not hurt me': 7, 'do not make me angry': 7, 'do not take it to the heart': 7, 'do not worry': 7, 'do you need something': 7, 'had your food': 7, 'he came by train': 7, 'he she is my friend': 7, 'he would be coming today': 7, 'help me': 7, 'hi how are you': 7, 'how can i help you': 7, 'how can i trust you': 7, 'how dare you': 7, 'how old are you': 7, 'i am afraid of that': 7, 'i am crying': 7, 'i am feeling bored': 7, 'i am feeling cold': 7, 'i am fine. thank you sir': 7, 'i am hungry': 7, 'i am in 

C:\Users\Legion\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Legion\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Legion\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [11]:
import os
import numpy as np
import csv
from collections import Counter

# === CONFIG ===
NpyFolder = "ML/Hand_LandMark"
LabelsFile = "labels.csv"

# === Load all data ===
X_all = []
y_all = []

with open(LabelsFile, mode="r") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        npy_filename, label = row
        npy_path = os.path.join(NpyFolder, npy_filename)

        if not os.path.exists(npy_path):
            print(f"❌ Missing file: {npy_path}")
            continue

        features = np.load(npy_path)
        X_all.append(features)
        y_all.append(label)

print("✅ Loaded total samples:", len(X_all))
print("✅ Label distribution:", Counter(y_all))

# === Select Top 5 Labels ===
label_counts = Counter(y_all)
top5_labels = [label for label, count in label_counts.most_common(5)]
print("✅ Top 5 labels:", top5_labels)

# === Keep only samples with those labels ===
X_top5 = []
y_top5 = []

for features, label in zip(X_all, y_all):
    if label in top5_labels:
        X_top5.append(features)
        y_top5.append(label)

print("✅ New dataset size:", len(X_top5))
print("✅ New label distribution:", Counter(y_top5))

# === OPTIONAL: Inspect a sample ===
print("✅ Example feature shape:", X_top5[0].shape)
print("✅ Example label:", y_top5[0])


✅ Loaded total samples: 687
✅ Label distribution: Counter({'serve the food': 9, 'can i help you': 8, 'go and sleep': 8, 'i enjoyed a lot': 8, 'i like you i love you': 8, 'speak softly': 8, 'you are bad': 8, 'He is going into the room': 7, 'This place is beautiful': 7, 'are you free today': 7, 'are you hiding something': 7, 'bring water for me': 7, 'can you repeat that please': 7, 'comb your hair': 7, 'congratulations': 7, 'could you please talk slower': 7, 'do me a favour': 7, 'do not hurt me': 7, 'do not make me angry': 7, 'do not take it to the heart': 7, 'do not worry': 7, 'do you need something': 7, 'had your food': 7, 'he came by train': 7, 'he she is my friend': 7, 'he would be coming today': 7, 'help me': 7, 'hi how are you': 7, 'how can i help you': 7, 'how can i trust you': 7, 'how dare you': 7, 'how old are you': 7, 'i am afraid of that': 7, 'i am crying': 7, 'i am feeling bored': 7, 'i am feeling cold': 7, 'i am fine. thank you sir': 7, 'i am hungry': 7, 'i am in dilemma wha

In [14]:
from collections import Counter

# Get the first label name
first_label = y_top5[0]

# Count all labels
label_counts = Counter(y_top5)

# Print info
print("✅ First label:", first_label)
print("✅ Number of samples for this label:", label_counts[first_label])


✅ First label: can i help you
✅ Number of samples for this label: 8


In [22]:
import os
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report
import csv

# === CONFIG ===
NpyFolder = "ML/Hand_LandMark"
LabelsFile = "labels.csv"

# === Load Data ===
X = []
y = []

with open(LabelsFile, mode="r") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        if len(row) < 2:
            print("⚠️ Skipping invalid row:", row)
            continue
        npy_filename = row[0].strip()
        label = row[1].strip()
        npy_path = os.path.join(NpyFolder, npy_filename)

        if not os.path.exists(npy_path):
            print(f"⚠️ File not found: {npy_path}")
            continue

        features = np.load(npy_path)
        X.append(features)
        y.append(label)

print("✅ Loaded samples:", len(X))
print("✅ Label counts before filtering:", Counter(y))
print()

# === Remove classes with <2 samples ===
label_counts = Counter(y)
X_filtered = []
y_filtered = []

for xi, yi in zip(X, y):
    if label_counts[yi] >= 2:
        X_filtered.append(xi)
        y_filtered.append(yi)

print("✅ Samples after filtering:", len(X_filtered))
print("✅ Label counts after filtering:", Counter(y_filtered))
print()

# === Encode labels ===
le = LabelEncoder()
y_encoded = le.fit_transform(y_filtered)

print("✅ Encoded labels:", y_encoded)
print("✅ Classes:", list(le.classes_))
print()

# === Train/Test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered,
    y_encoded,
    test_size=0.3,
    stratify=y_encoded,
    random_state=42
)

print("✅ Train size:", len(X_train))
print("✅ Test size:", len(X_test))
print()

# === Train classifier ===
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# === Predict ===
y_pred = knn.predict(X_test)

# === Evaluate ===
print("🎯 Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))


✅ Loaded samples: 687
✅ Label counts before filtering: Counter({'serve the food': 9, 'can i help you': 8, 'go and sleep': 8, 'i enjoyed a lot': 8, 'i like you i love you': 8, 'speak softly': 8, 'you are bad': 8, 'He is going into the room': 7, 'This place is beautiful': 7, 'are you free today': 7, 'are you hiding something': 7, 'bring water for me': 7, 'can you repeat that please': 7, 'comb your hair': 7, 'congratulations': 7, 'could you please talk slower': 7, 'do me a favour': 7, 'do not hurt me': 7, 'do not make me angry': 7, 'do not take it to the heart': 7, 'do not worry': 7, 'do you need something': 7, 'had your food': 7, 'he came by train': 7, 'he she is my friend': 7, 'he would be coming today': 7, 'help me': 7, 'hi how are you': 7, 'how can i help you': 7, 'how can i trust you': 7, 'how dare you': 7, 'how old are you': 7, 'i am afraid of that': 7, 'i am crying': 7, 'i am feeling bored': 7, 'i am feeling cold': 7, 'i am fine. thank you sir': 7, 'i am hungry': 7, 'i am in dilemm

C:\Users\Legion\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Legion\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Legion\miniconda3\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [23]:
print("✅ Accuracy: {:.2f}%".format(acc * 100))

✅ Accuracy: 2.92%


In [1]:
import os
import numpy as np
from collections import Counter
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, accuracy_score
import csv

# === CONFIG ===
NpyFolder = "ML/Hand_LandMark"
LabelsFile = "labels.csv"
NUM_AUGMENTATIONS = 5  # how many times to augment each sample

# === Load and augment ===
X = []
y = []

def augment_landmarks(landmarks):
    noise = np.random.normal(0, 0.01, size=landmarks.shape)
    return landmarks + noise

with open(LabelsFile, mode="r") as f:
    reader = csv.reader(f)
    next(reader)  # skip header
    for row in reader:
        if len(row) < 2:
            print("⚠️ Skipping invalid row:", row)
            continue
        npy_filename = row[0].strip()
        label = row[1].strip()
        npy_path = os.path.join(NpyFolder, npy_filename)

        if not os.path.exists(npy_path):
            print(f"⚠️ File not found: {npy_path}")
            continue

        features = np.load(npy_path)

        # Original sample
        X.append(features)
        y.append(label)

        # Augmented samples
        for i in range(NUM_AUGMENTATIONS):
            augmented = augment_landmarks(features)
            X.append(augmented)
            y.append(label)

print("✅ Loaded + augmented samples:", len(X))
print("✅ Label counts:", Counter(y))
print()

# === Remove classes with <2 samples ===
label_counts = Counter(y)
X_filtered = []
y_filtered = []

for xi, yi in zip(X, y):
    if label_counts[yi] >= 2:
        X_filtered.append(xi)
        y_filtered.append(yi)

print("✅ Samples after filtering:", len(X_filtered))
print("✅ Label counts after filtering:", Counter(y_filtered))
print()

# === Encode labels ===
le = LabelEncoder()
y_encoded = le.fit_transform(y_filtered)

# === Train/Test split ===
X_train, X_test, y_train, y_test = train_test_split(
    X_filtered,
    y_encoded,
    test_size=0.3,
    stratify=y_encoded,
    random_state=42
)

# === Train classifier ===
knn = KNeighborsClassifier(n_neighbors=3)
knn.fit(X_train, y_train)

# === Predict ===
y_pred = knn.predict(X_test)

# === Evaluate ===
acc = accuracy_score(y_test, y_pred)
print("🎯 Classification Report:")
print(classification_report(y_test, y_pred, target_names=le.classes_))
print("✅ Accuracy: {:.2f}%".format(acc * 100))


✅ Loaded + augmented samples: 4122
✅ Label counts: Counter({'serve the food': 54, 'can i help you': 48, 'go and sleep': 48, 'i enjoyed a lot': 48, 'i like you i love you': 48, 'speak softly': 48, 'you are bad': 48, 'He is going into the room': 42, 'This place is beautiful': 42, 'are you free today': 42, 'are you hiding something': 42, 'bring water for me': 42, 'can you repeat that please': 42, 'comb your hair': 42, 'congratulations': 42, 'could you please talk slower': 42, 'do me a favour': 42, 'do not hurt me': 42, 'do not make me angry': 42, 'do not take it to the heart': 42, 'do not worry': 42, 'do you need something': 42, 'had your food': 42, 'he came by train': 42, 'he she is my friend': 42, 'he would be coming today': 42, 'help me': 42, 'hi how are you': 42, 'how can i help you': 42, 'how can i trust you': 42, 'how dare you': 42, 'how old are you': 42, 'i am afraid of that': 42, 'i am crying': 42, 'i am feeling bored': 42, 'i am feeling cold': 42, 'i am fine. thank you sir': 42, 

In [25]:
import numpy as np

# === Path to your single test file ===
test_npy = "ML/Hand_LandMark/can i help you_help.npy"

# Load the features
features = np.load(test_npy)

# Predict
predicted_label_index = knn.predict([features])[0]
predicted_label = le.inverse_transform([predicted_label_index])[0]

print("🎯 Predicted Label:", predicted_label)


🎯 Predicted Label: can i help you


In [26]:
import joblib

# Save KNN model
joblib.dump(knn, "models/knn_model.joblib")

# Save LabelEncoder (so you can decode predictions)
joblib.dump(le, "models/label_encoder.joblib")


['label_encoder.joblib']

In [2]:
import os
import shutil
import pandas as pd

# ===========================
# PATHS
# ===========================
SAVE_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/image_data"
LABELS_CSV = os.path.join(SAVE_PATH, "labels.csv")

# ===========================
# LOAD LABEL DATA
# ===========================
df = pd.read_csv(LABELS_CSV)

removed_labels = []

# ===========================
# LOOP THROUGH LABEL FOLDERS
# ===========================
for label in os.listdir(SAVE_PATH):
    label_path = os.path.join(SAVE_PATH, label)
    if not os.path.isdir(label_path):
        continue

    npy_files = [f for f in os.listdir(label_path) if f.endswith(".npy")]

    if len(npy_files) < 3:
        print(f"🗑 Removing label '{label}' ({len(npy_files)} samples)")
        shutil.rmtree(label_path)
        removed_labels.append(label)

# ===========================
# UPDATE LABEL CSV
# ===========================
if removed_labels:
    df = df[~df['label'].isin(removed_labels)]
    df.to_csv(LABELS_CSV, index=False)
    print(f"✅ Updated labels.csv (removed {len(removed_labels)} labels)")
else:
    print("✅ No folders had fewer than 3 samples.")

print("Cleanup complete.")


🗑 Removing label 'A LOT' (2 samples)
🗑 Removing label 'ABUSE' (1 samples)
🗑 Removing label 'AFRAID' (0 samples)
🗑 Removing label 'AGREE' (2 samples)
🗑 Removing label 'ALL' (1 samples)
🗑 Removing label 'ANGRY' (2 samples)
🗑 Removing label 'ANYTHING' (2 samples)
🗑 Removing label 'BAD' (1 samples)
🗑 Removing label 'BEAUTIFUL' (2 samples)
🗑 Removing label 'BED' (2 samples)
🗑 Removing label 'BORED' (1 samples)
🗑 Removing label 'CHAT' (1 samples)
🗑 Removing label 'CLASS' (1 samples)
🗑 Removing label 'COLD' (2 samples)
🗑 Removing label 'COMB' (2 samples)
🗑 Removing label 'COME' (1 samples)
🗑 Removing label 'DARE' (1 samples)
🗑 Removing label 'DIFFERENCE' (1 samples)
🗑 Removing label 'DISAPPOINTED' (1 samples)
🗑 Removing label 'DON'T CARE' (0 samples)
🗑 Removing label 'FAVOUR' (1 samples)
🗑 Removing label 'FEVER' (2 samples)
🗑 Removing label 'FREE' (2 samples)
🗑 Removing label 'FROM' (2 samples)
🗑 Removing label 'GOOD' (2 samples)
🗑 Removing label 'GRATEFUL' (2 samples)
🗑 Removing label 'HAD' 

In [3]:
import os
import numpy as np
import pandas as pd
from tqdm import tqdm
import random

# ===========================
# PATHS
# ===========================
SAVE_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/image_data"
LABELS_CSV = os.path.join(SAVE_PATH, "labels.csv")

# ===========================
# LOAD LABEL CSV
# ===========================
df = pd.read_csv(LABELS_CSV)

# ===========================
# AUGMENTATION FUNCTIONS
# ===========================
def random_noise(landmarks):
    noise = np.random.normal(0, 0.005, landmarks.shape)
    return landmarks + noise

def random_scale(landmarks):
    scale = np.random.uniform(0.95, 1.05)
    return landmarks * scale

def random_flip(landmarks):
    # Flip along x-axis
    flipped = landmarks.copy()
    flipped[0::3] = 1 - flipped[0::3]  # Invert x values (0–1 range)
    return flipped

def random_shift(landmarks):
    shift_x = np.random.uniform(-0.02, 0.02)
    shift_y = np.random.uniform(-0.02, 0.02)
    shifted = landmarks.copy()
    shifted[0::3] = np.clip(shifted[0::3] + shift_x, 0, 1)
    shifted[1::3] = np.clip(shifted[1::3] + shift_y, 0, 1)
    return shifted

augmentations = [random_noise, random_scale, random_flip, random_shift]

# ===========================
# FIND MAX COUNT
# ===========================
label_counts = {label: len([f for f in os.listdir(os.path.join(SAVE_PATH, label)) if f.endswith(".npy")])
                for label in os.listdir(SAVE_PATH) if os.path.isdir(os.path.join(SAVE_PATH, label))}

max_count = max(label_counts.values())
print(f"📊 Maximum samples found: {max_count}")

# ===========================
# BALANCE EACH LABEL
# ===========================
for label, count in tqdm(label_counts.items(), desc="Augmenting data"):
    label_path = os.path.join(SAVE_PATH, label)
    npy_files = [f for f in os.listdir(label_path) if f.endswith(".npy")]

    if count >= max_count:
        continue

    needed = max_count - count
    existing_data = [np.load(os.path.join(label_path, f)) for f in npy_files]
    generated = 0

    while generated < needed:
        base = random.choice(existing_data)
        aug_func = random.choice(augmentations)
        new_data = aug_func(base)

        new_name = f"aug_{generated}_{random.randint(1000,9999)}.npy"
        np.save(os.path.join(label_path, new_name), new_data)

        df = pd.concat([df, pd.DataFrame({"filename": [new_name], "label": [label]})], ignore_index=True)
        generated += 1

# ===========================
# SAVE UPDATED LABEL FILE
# ===========================
df.to_csv(LABELS_CSV, index=False)

print("✅ Augmentation complete.")
print("All folders now have the same number of samples as the largest class!")


📊 Maximum samples found: 36


Augmenting data: 100%|██████████| 41/41 [00:33<00:00,  1.21it/s]

✅ Augmentation complete.
All folders now have the same number of samples as the largest class!


In [5]:
import os
import numpy as np
import pandas as pd
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.model_selection import train_test_split
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Dropout
from tensorflow.keras.utils import to_categorical

# ===========================
# PATHS
# ===========================
DATA_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/image_data"
LABELS_CSV = os.path.join(DATA_PATH, "labels.csv")

# ===========================
# LOAD DATA
# ===========================
df = pd.read_csv(LABELS_CSV)

X, y = [], []
for _, row in df.iterrows():
    npy_path = os.path.join(DATA_PATH, row["label"], row["filename"])
    if os.path.exists(npy_path):
        X.append(np.load(npy_path))
        y.append(row["label"])

X = np.array(X)
y = np.array(y)

# ===========================
# ENCODE LABELS + ONE-HOT
# ===========================
le = LabelEncoder()
y_encoded = le.fit_transform(y)
y_cat = to_categorical(y_encoded)

# ===========================
# SCALE FEATURES
# ===========================
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# ===========================
# TRAIN-TEST SPLIT
# ===========================
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y_cat, test_size=0.2, stratify=y_encoded, random_state=42
)

# ===========================
# BUILD MLP MODEL
# ===========================
num_classes = y_cat.shape[1]
model = Sequential([
    Dense(128, activation='relu', input_shape=(X_train.shape[1],)),
    Dropout(0.3),
    Dense(64, activation='relu'),
    Dropout(0.3),
    Dense(num_classes, activation='softmax')
])

model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# ===========================
# TRAIN MODEL
# ===========================
history = model.fit(
    X_train, y_train,
    validation_data=(X_test, y_test),
    epochs=100,
    batch_size=32
)

# ===========================
# EVALUATE
# ===========================
loss, acc = model.evaluate(X_test, y_test)
print(f"\n✅ Test Accuracy: {acc*100:.2f}%")

# ===========================
# SAVE MODEL + SCALER + LABEL ENCODER
# ===========================
model.save(os.path.join(DATA_PATH, "hand_sign_mlp_model.h5"))

import joblib
joblib.dump(scaler, os.path.join(DATA_PATH, "scaler.save"))
joblib.dump(le, os.path.join(DATA_PATH, "label_encoder.save"))

print("✅ Model, scaler, and label encoder saved for inference!")


C:\Users\Legion\AppData\Local\Programs\Python\Python310\lib\site-packages\keras\src\layers\core\dense.py:92: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ dense_3 (Dense)                 │ (None, 128)            │         8,192 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_2 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 64)             │         8,256 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_3 (Dropout)             │ (None, 64)             │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ (None, 41)             │         2,665 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 19,113 (74.66 KB)

 Trainable params: 19,113 (74.66 KB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 8s 40ms/step - accuracy: 0.0407 - loss: 3.7554 - val_accuracy: 0.1216 - val_loss: 3.4740
Epoch 2/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.0822 - loss: 3.5136 - val_accuracy: 0.1622 - val_loss: 3.3368
Epoch 3/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.0941 - loss: 3.4020 - val_accuracy: 0.1858 - val_loss: 3.1903
Epoch 4/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - accuracy: 0.1314 - loss: 3.2500 - val_accuracy: 0.2061 - val_loss: 3.0350
Epoch 5/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 21ms/step - accuracy: 0.1534 - loss: 3.1335 - val_accuracy: 0.2703 - val_loss: 2.9033
Epoch 6/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 19ms/step - accuracy: 0.1720 - loss: 3.0002 - val_accuracy: 0.2770 - val_loss: 2.7744
Epoch 7/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 22ms/step - accuracy: 0.2102 - loss: 2.8604 - val_accuracy: 0.3074 - val_loss: 2.6614
Epoch 8/100
37/37 ━━━━━━━━━━━━━━━━━━━━ 1s 20ms/step - accuracy: 0.2178 - loss: 2.7653 - val_accuracy: 0.


✅ Model, scaler, and label encoder saved for inference!


In [8]:
import cv2
import mediapipe as mp
import os
import numpy as np
from tqdm import tqdm

DATA_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/test"
SAVE_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/video_data"
os.makedirs(SAVE_PATH, exist_ok=True)

mp_holistic = mp.solutions.holistic
holistic = mp_holistic.Holistic(static_image_mode=False,
                                min_detection_confidence=0.5,
                                min_tracking_confidence=0.5)

def extract_landmarks(frame):
    # Convert to grayscale
    gray = cv2.cvtColor(frame, cv2.COLOR_BGR2GRAY)
    rgb = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)

    results = holistic.process(rgb)
    # Skip frame if no hands detected at all
    if not results.left_hand_landmarks and not results.right_hand_landmarks:
        return None

    landmarks = []

    # Pose for relative coordinates
    if results.pose_landmarks:
        l_shoulder = results.pose_landmarks.landmark[11]
        r_shoulder = results.pose_landmarks.landmark[12]
        shoulder_center = np.array([(l_shoulder.x+r_shoulder.x)/2,
                                    (l_shoulder.y+r_shoulder.y)/2,
                                    (l_shoulder.z+r_shoulder.z)/2])
    else:
        shoulder_center = np.zeros(3)

    # Left & Right Hands
    for hand in [results.left_hand_landmarks, results.right_hand_landmarks]:
        if hand:
            for lm in hand.landmark:
                relative = np.array([lm.x, lm.y, lm.z]) - shoulder_center
                landmarks.extend(relative)
        else:
            landmarks.extend([0]*63)  # 21*3 per hand

    # Pose landmarks (optional)
    if results.pose_landmarks:
        for lm in results.pose_landmarks.landmark:
            landmarks.extend([lm.x, lm.y, lm.z])
    else:
        landmarks.extend([0]*99)  # 33*3 for pose

    return np.array(landmarks, dtype=np.float32)

# Process all videos
labels = os.listdir(DATA_PATH)
for label in labels:
    label_path = os.path.join(DATA_PATH, label)
    save_label_path = os.path.join(SAVE_PATH, label)
    os.makedirs(save_label_path, exist_ok=True)

    for vid_file in tqdm(os.listdir(label_path)):
        if not vid_file.endswith(('.mp4', '.avi')):
            continue

        cap = cv2.VideoCapture(os.path.join(label_path, vid_file))
        seq = []
        hand_detected = False

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            lm = extract_landmarks(frame)
            if lm is not None:
                hand_detected = True  # At least one frame has a hand
                seq.append(lm)

        cap.release()

        # Remove video if no hands detected in entire video
        if not hand_detected or len(seq) == 0:
            print(f"🗑 Removing video {vid_file} (no hands detected in any frame)")
            continue

        seq = np.array(seq)
        np.save(os.path.join(save_label_path, os.path.splitext(vid_file)[0]+".npy"), seq)

holistic.close()
print("✅ Video preprocessing complete — videos with no hands removed!")


100%|██████████| 7/7 [00:17<00:00,  2.52s/it]

✅ Video preprocessing complete — videos with no hands removed!


In [10]:
import os
import numpy as np
from tqdm import tqdm

DATA_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/video_data"
AUG_SAVE_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/video_augmented"
os.makedirs(AUG_SAVE_PATH, exist_ok=True)

def normalize_sequence(seq):
    """Normalize sequence values to [-1, 1] based on max absolute value"""
    max_val = np.max(np.abs(seq))
    if max_val > 0:
        seq = seq / max_val
    return seq

def augment_sequence(seq):
    aug_seqs = []
    for i in range(30):  # generate 30 augmented sequences
        new_seq = seq.copy()

        # 1. Small rotation around z-axis
        angle = np.random.uniform(-10, 10) * np.pi/180
        rot_matrix = np.array([[np.cos(angle), -np.sin(angle), 0],
                               [np.sin(angle),  np.cos(angle), 0],
                               [0, 0, 1]])
        for f in range(new_seq.shape[0]):
            coords = new_seq[f].reshape(-1, 3)
            coords = coords @ rot_matrix.T
            new_seq[f] = coords.flatten()

        # 2. Scaling
        scale = np.random.uniform(0.9, 1.1)
        new_seq *= scale

        # 3. Translation
        shift = np.random.uniform(-0.05, 0.05, size=(1, new_seq.shape[1]))
        new_seq += shift

        # 4. Gaussian noise
        noise = np.random.normal(0, 0.005, size=new_seq.shape)
        new_seq += noise

        # 5. Time warping (slight speed-up/slow-down)
        factor = np.random.uniform(0.9, 1.1)
        idxs = np.clip((np.arange(0, new_seq.shape[0], 1/factor)).astype(int), 0, new_seq.shape[0]-1)
        new_seq = new_seq[idxs]

        # 6. Normalize
        new_seq = normalize_sequence(new_seq)

        aug_seqs.append(new_seq)
    return aug_seqs

# Process all labels
labels = os.listdir(DATA_PATH)
for label in labels:
    label_path = os.path.join(DATA_PATH, label)
    save_label_path = os.path.join(AUG_SAVE_PATH, label)
    os.makedirs(save_label_path, exist_ok=True)

    for file in tqdm(os.listdir(label_path)):
        if not file.endswith(".npy"):
            continue
        seq = np.load(os.path.join(label_path, file))
        aug_seqs = augment_sequence(seq)
        base_name = os.path.splitext(file)[0]
        for i, aug_seq in enumerate(aug_seqs):
            np.save(os.path.join(save_label_path, f"{base_name}_aug{i}.npy"), aug_seq)

print("✅ Augmentation + normalization complete — 30 sequences per `.npy` generated!")


100%|██████████| 2/2 [00:00<00:00,  4.83it/s]

✅ Augmentation + normalization complete — 30 sequences per `.npy` generated!


In [11]:
import os
import numpy as np
from tensorflow.keras.utils import to_categorical
from sklearn.model_selection import train_test_split
from tqdm import tqdm

DATA_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/video_augmented"
SEQ_LENGTH = 50  # max frames per sequence

# 1️⃣ Prepare label mapping
labels_list = sorted(os.listdir(DATA_PATH))
num_classes = len(labels_list)
label_map = {label: i for i, label in enumerate(labels_list)}
print("Labels mapping:", label_map)

X = []
y = []

# 2️⃣ Load sequences, pad/truncate, and prepare labels
for label in labels_list:
    label_path = os.path.join(DATA_PATH, label)
    for file in tqdm(os.listdir(label_path), desc=f"Processing {label}"):
        if not file.endswith(".npy"):
            continue
        seq = np.load(os.path.join(label_path, file))
        # Pad or truncate sequence
        if seq.shape[0] < SEQ_LENGTH:
            pad_len = SEQ_LENGTH - seq.shape[0]
            pad_seq = np.vstack([seq, np.tile(seq[-1], (pad_len, 1))])
        else:
            pad_seq = seq[:SEQ_LENGTH]
        X.append(pad_seq)
        y.append(label_map[label])

X = np.array(X, dtype=np.float32)
y = to_categorical(np.array(y), num_classes=num_classes)

# 3️⃣ Train/Validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train: {X_train.shape}, X_val: {X_val.shape}")
print(f"y_train: {y_train.shape}, y_val: {y_val.shape}")


Labels mapping: {'are you hiding something': 0, 'can i help you': 1, 'can you repeat that please': 2, 'do me a favour': 3, 'do not worry': 4, 'do you need something': 5, 'help me': 6, 'hi how are you': 7, 'how can i help you': 8, 'how can i trust you': 9, 'what are you doing': 10, 'what happened': 11, 'what you want': 12, 'you are good': 13}


Processing you are good: 100%|██████████| 60/60 [00:02<00:00, 28.62it/s]


X_train: (648, 50, 225), X_val: (162, 50, 225)
y_train: (648, 14), y_val: (162, 14)


In [12]:
import tensorflow as tf
from tensorflow.keras import layers, models

# 1️⃣ Model parameters
SEQ_LENGTH = X_train.shape[1]    # 50 frames (or whatever your preprocessing padded to)
NUM_FEATURES = X_train.shape[2]  # number of landmarks per frame
NUM_CLASSES = y_train.shape[1]

# 2️⃣ Input
inputs = layers.Input(shape=(SEQ_LENGTH, NUM_FEATURES))

# LSTM branch
x1 = layers.LSTM(128, return_sequences=True)(inputs)
x1 = layers.Dropout(0.2)(x1)

# GRU branch
x2 = layers.GRU(128, return_sequences=True)(inputs)
x2 = layers.Dropout(0.2)(x2)

# Combine LSTM + GRU
x = layers.Concatenate()([x1, x2])

# Transformer block
def transformer_block(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.1):
    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(x, x)
    attn_output = layers.Dropout(dropout)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)

    ffn = layers.Dense(ff_dim, activation='relu')(out1)
    ffn = layers.Dense(x.shape[-1])(ffn)
    ffn = layers.Dropout(dropout)(ffn)
    return layers.LayerNormalization(epsilon=1e-6)(out1 + ffn)

x = transformer_block(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(NUM_CLASSES, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# 3️⃣ Callbacks
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

# 4️⃣ Training
history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=100,
    batch_size=16,
    callbacks=[early_stop]
)


Model: "functional_7"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_2       │ (None, 50, 225)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm (LSTM)         │ (None, 50, 128)   │    181,248 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 50, 128)   │    136,320 │ input_layer_2[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_4 (Dropout) │ (None, 50, 128)   │          0 │ lstm[0][0]        │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_5 (Dropout) │ (None, 50, 128)   │          0 │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate         │ (None, 50, 256)   │          0 │ dropout_4[0][0],  │
│ (Concatenate)       │                   │            │ dropout_5[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 256)   │    263,168 │ concatenate[0][0… │
│ (MultiHeadAttentio… │                   │            │ concatenate[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_7 (Dropout) │ (None, 50, 256)   │          0 │ multi_head_atten… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 256)   │          0 │ concatenate[0][0… │
│                     │                   │            │ dropout_7[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalization │ (None, 50, 256)   │        512 │ add[0][0]         │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_6 (Dense)     │ (None, 50, 128)   │     32,896 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_7 (Dense)     │ (None, 50, 256)   │     33,024 │ dense_6[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_8 (Dropout) │ (None, 50, 256)   │          0 │ dense_7[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_1 (Add)         │ (None, 50, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_8[0][0]   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 256)   │        512 │ add_1[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_8 (Dense)     │ (None, 128)       │     32,896 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_9 (Dropout) │ (None, 128)       │          0 │ dense_8[0][0]     │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_9 (Dense)     │ (None, 14)        │      1,806 │ dropout_9[0][0]   │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 682,382 (2.60 MB)

 Trainable params: 682,382 (2.60 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
41/41 ━━━━━━━━━━━━━━━━━━━━ 58s 572ms/step - accuracy: 0.0818 - loss: 2.7536 - val_accuracy: 0.3086 - val_loss: 2.4980
Epoch 2/100
41/41 ━━━━━━━━━━━━━━━━━━━━ 12s 303ms/step - accuracy: 0.2145 - loss: 2.3016 - val_accuracy: 0.3519 - val_loss: 1.9487
Epoch 3/100
41/41 ━━━━━━━━━━━━━━━━━━━━ 18s 443ms/step - accuracy: 0.3904 - loss: 1.7107 - val_accuracy: 0.6173 - val_loss: 1.0571
Epoch 4/100
41/41 ━━━━━━━━━━━━━━━━━━━━ 14s 338ms/step - accuracy: 0.6821 - loss: 0.8804 - val_accuracy: 0.9383 - val_loss: 0.2948
Epoch 5/100
41/41 ━━━━━━━━━━━━━━━━━━━━ 10s 242ms/step - accuracy: 0.8704 - loss: 0.4303 - val_accuracy: 1.0000 - val_loss: 0.0707
Epoch 6/100
41/41 ━━━━━━━━━━━━━━━━━━━━ 9s 216ms/step - accuracy: 0.9614 - loss: 0.1344 - val_accuracy: 1.0000 - val_loss: 0.0112
Epoch 7/100
41/41 ━━━━━━━━━━━━━━━━━━━━ 9s 213ms/step - accuracy: 0.9954 - loss: 0.0416 - val_accuracy: 1.0000 - val_loss: 0.0026
Epoch 8/100
41/41 ━━━━━━━━━━━━━━━━━━━━ 9s 218ms/step - accuracy: 0.9861 - loss: 0.0549 - val

In [13]:
# Print final training and validation accuracy
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]

print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {final_val_acc*100:.2f}%")


✅ Final Training Accuracy: 100.00%
✅ Final Validation Accuracy: 100.00%


In [15]:
import os
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import tensorflow as tf
from tensorflow.keras import layers, models

# =========================
# PARAMETERS
# =========================
DATA_PATH = r"D:/archive/ISL_CSLRT_Corpus/ISL_CSLRT_Corpus/video_data"  # your augmented landmark npy files
SEQ_LENGTH = 50      # max frames per sequence
AUG_PER_SEQ = 30     # augmentations per training sequence
BATCH_SIZE = 16
EPOCHS = 100

# =========================
# LABELS
# =========================
labels_list = sorted(os.listdir(DATA_PATH))
num_classes = len(labels_list)
label_map = {label: i for i, label in enumerate(labels_list)}
print("Labels mapping:", label_map)

# =========================
# AUGMENTATION FUNCTIONS
# =========================
def normalize_sequence(seq):
    max_val = np.max(np.abs(seq))
    return seq / max_val if max_val > 0 else seq

def augment_sequence(seq, num_aug=AUG_PER_SEQ):
    aug_seqs = []
    for _ in range(num_aug):
        new_seq = seq.copy()
        # Rotation
        angle = np.random.uniform(-10, 10) * np.pi/180
        rot = np.array([[np.cos(angle), -np.sin(angle),0],
                        [np.sin(angle), np.cos(angle),0],
                        [0,0,1]])
        for f in range(new_seq.shape[0]):
            coords = new_seq[f].reshape(-1,3)
            coords = coords @ rot.T
            new_seq[f] = coords.flatten()
        # Scale
        new_seq *= np.random.uniform(0.9,1.1)
        # Translate
        new_seq += np.random.uniform(-0.05,0.05, size=(1,new_seq.shape[1]))
        # Noise
        new_seq += np.random.normal(0,0.005,new_seq.shape)
        # Time warp
        factor = np.random.uniform(0.9,1.1)
        idxs = np.clip((np.arange(0,new_seq.shape[0],1/factor)).astype(int),0,new_seq.shape[0]-1)
        new_seq = new_seq[idxs]
        # Normalize
        new_seq = normalize_sequence(new_seq)
        aug_seqs.append(new_seq)
    return aug_seqs

# =========================
# LOAD DATA + AUGMENT
# =========================
X_train, y_train, X_val, y_val = [], [], [], []

for label in labels_list:
    label_path = os.path.join(DATA_PATH, label)
    files = [f for f in os.listdir(label_path) if f.endswith(".npy")]

    # Handle small number of sequences
    if len(files) < 2:
        train_files = files
        val_files = []
    else:
        train_files, val_files = train_test_split(files, test_size=0.2, random_state=42, stratify=None)

    # Training sequences + augmentation
    for file in train_files:
        seq = np.load(os.path.join(label_path, file))
        aug_seqs = augment_sequence(seq, AUG_PER_SEQ)
        for aug_seq in aug_seqs:
            if aug_seq.shape[0] < SEQ_LENGTH:
                pad_len = SEQ_LENGTH - aug_seq.shape[0]
                aug_seq = np.vstack([aug_seq, np.tile(aug_seq[-1], (pad_len,1))])
            else:
                aug_seq = aug_seq[:SEQ_LENGTH]
            X_train.append(aug_seq)
            y_train.append(label_map[label])

    # Validation sequences (no augmentation)
    for file in val_files:
        seq = np.load(os.path.join(label_path, file))
        if seq.shape[0] < SEQ_LENGTH:
            pad_len = SEQ_LENGTH - seq.shape[0]
            seq = np.vstack([seq, np.tile(seq[-1], (pad_len,1))])
        else:
            seq = seq[:SEQ_LENGTH]
        X_val.append(seq)
        y_val.append(label_map[label])

# Convert to numpy arrays
X_train = np.array(X_train, dtype=np.float32)
y_train = to_categorical(np.array(y_train), num_classes=num_classes)
X_val = np.array(X_val, dtype=np.float32)
y_val = to_categorical(np.array(y_val), num_classes=num_classes)

print(f"✅ Dataset ready: X_train={X_train.shape}, X_val={X_val.shape}")

# =========================
# HYBRID MODEL: LSTM + GRU + TRANSFORMER
# =========================
inputs = layers.Input(shape=(SEQ_LENGTH, X_train.shape[2]))

# LSTM branch
x1 = layers.LSTM(128, return_sequences=True)(inputs)
x1 = layers.Dropout(0.2)(x1)

# GRU branch
x2 = layers.GRU(128, return_sequences=True)(inputs)
x2 = layers.Dropout(0.2)(x2)

# Concatenate
x = layers.Concatenate()([x1, x2])

# Transformer block
def transformer_block(x, head_size=64, num_heads=4, ff_dim=128, dropout=0.1):
    attn_output = layers.MultiHeadAttention(num_heads=num_heads, key_dim=head_size)(x, x)
    attn_output = layers.Dropout(dropout)(attn_output)
    out1 = layers.LayerNormalization(epsilon=1e-6)(x + attn_output)
    ffn = layers.Dense(ff_dim, activation='relu')(out1)
    ffn = layers.Dense(x.shape[-1])(ffn)
    ffn = layers.Dropout(dropout)(ffn)
    return layers.LayerNormalization(epsilon=1e-6)(out1 + ffn)

x = transformer_block(x)
x = layers.GlobalAveragePooling1D()(x)
x = layers.Dense(128, activation='relu')(x)
x = layers.Dropout(0.3)(x)
outputs = layers.Dense(num_classes, activation='softmax')(x)

model = models.Model(inputs, outputs)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()

# =========================
# TRAINING
# =========================
early_stop = tf.keras.callbacks.EarlyStopping(monitor='val_loss', patience=10, restore_best_weights=True)

history = model.fit(
    X_train, y_train,
    validation_data=(X_val, y_val),
    epochs=EPOCHS,
    batch_size=BATCH_SIZE,
    callbacks=[early_stop]
)

# =========================
# PRINT FINAL ACCURACY
# =========================
final_train_acc = history.history['accuracy'][-1]
final_val_acc = history.history['val_accuracy'][-1]

print(f"✅ Final Training Accuracy: {final_train_acc*100:.2f}%")
print(f"✅ Final Validation Accuracy: {final_val_acc*100:.2f}%")


Labels mapping: {'are you hiding something': 0, 'can i help you': 1, 'can you repeat that please': 2, 'do me a favour': 3, 'do not worry': 4, 'do you need something': 5, 'help me': 6, 'hi how are you': 7, 'how can i help you': 8, 'how can i trust you': 9, 'what are you doing': 10, 'what happened': 11, 'what you want': 12, 'you are good': 13}
✅ Dataset ready: X_train=(420, 50, 225), X_val=(13, 50, 225)


Model: "functional_8"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer_3       │ (None, 50, 225)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ lstm_1 (LSTM)       │ (None, 50, 128)   │    181,248 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 50, 128)   │    136,320 │ input_layer_3[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_10          │ (None, 50, 128)   │          0 │ lstm_1[0][0]      │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_11          │ (None, 50, 128)   │          0 │ gru_1[0][0]       │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ concatenate_1       │ (None, 50, 256)   │          0 │ dropout_10[0][0], │
│ (Concatenate)       │                   │            │ dropout_11[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ multi_head_attenti… │ (None, 50, 256)   │    263,168 │ concatenate_1[0]… │
│ (MultiHeadAttentio… │                   │            │ concatenate_1[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_13          │ (None, 50, 256)   │          0 │ multi_head_atten… │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_2 (Add)         │ (None, 50, 256)   │          0 │ concatenate_1[0]… │
│                     │                   │            │ dropout_13[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 256)   │        512 │ add_2[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_10 (Dense)    │ (None, 50, 128)   │     32,896 │ layer_normalizat… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_11 (Dense)    │ (None, 50, 256)   │     33,024 │ dense_10[0][0]    │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_14          │ (None, 50, 256)   │          0 │ dense_11[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add_3 (Add)         │ (None, 50, 256)   │          0 │ layer_normalizat… │
│                     │                   │            │ dropout_14[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ layer_normalizatio… │ (None, 50, 256)   │        512 │ add_3[0][0]       │
│ (LayerNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 256)       │          0 │ layer_normalizat… │
│ (GlobalAveragePool… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_12 (Dense)    │ (None, 128)       │     32,896 │ global_average_p… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dropout_15          │ (None, 128)       │          0 │ dense_12[0][0]    │
│ (Dropout)           │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼─────────────────

 Total params: 682,382 (2.60 MB)

 Trainable params: 682,382 (2.60 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 56s 557ms/step - accuracy: 0.0690 - loss: 2.8092 - val_accuracy: 0.0769 - val_loss: 2.6374
Epoch 2/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 7s 236ms/step - accuracy: 0.1476 - loss: 2.5030 - val_accuracy: 0.0769 - val_loss: 2.6725
Epoch 3/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 15s 572ms/step - accuracy: 0.1238 - loss: 2.5243 - val_accuracy: 0.0000e+00 - val_loss: 2.7382
Epoch 4/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 8s 255ms/step - accuracy: 0.2429 - loss: 1.9891 - val_accuracy: 0.3077 - val_loss: 2.9912
Epoch 5/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 8s 305ms/step - accuracy: 0.6571 - loss: 0.9713 - val_accuracy: 0.2308 - val_loss: 3.7744
Epoch 6/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 9s 319ms/step - accuracy: 0.8357 - loss: 0.4671 - val_accuracy: 0.3846 - val_loss: 4.2596
Epoch 7/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 8s 304ms/step - accuracy: 0.9143 - loss: 0.2959 - val_accuracy: 0.2308 - val_loss: 5.2320
Epoch 8/100
27/27 ━━━━━━━━━━━━━━━━━━━━ 9s 327ms/step - accuracy: 0.9905 - loss: 0.0632 - va